In [9]:
from gpzoo.gp import WSVGP
from gpzoo.kernels import NSF_RBF
from torch import nn
from torch import optim
from tqdm.autonotebook import tqdm
from torch import distributions
import torch

In [6]:
class DeepGP(nn.Module):
    def __init__(self, gp1, gp2, noise=0.1):
        super().__init__()
        self.gp1 = gp1
        self.gp2 = gp2

        self.noise = nn.Parameter(torch.tensor(noise))

    def forward(self, X, E=1, verbose=False, **kwargs):
        qF1, qU1, pU1 = self.gp1(X, verbose=verbose, **kwargs)
        
        X2 = qF1.resample().T    
        qF2, qU2, pU2 = self.gp2(X2, verbose=verbose, **kwargs)
        
        
        F2 = qF2.rsample((E, ))
        
        

        noise = torch.nn.functional.softplus(self.noise) #ensure positive
        pY = distributions.Normal(F2, noise)

        return pY, qF1, qU1, pU1, qF2, qU2, pU2
    
    
    def predict(self, X, verbose=False, **kwargs):
        qF1, qU1, pU1 = self.gp1(X, verbose=verbose, **kwargs)
        
        X2 = qF1.mean.T    
        qF2, qU2, pU2 = self.gp2(X2, verbose=verbose, **kwargs)
        
    

        return qF1, qU1, pU1, qF2, qU2, pU2

In [7]:
class CustomLikelihood(nn.Module):
    def __init__(self, gp1, gp2, noise=0.1):
        super().__init__()
        self.gp1 = gp1
        self.gp2 = gp2

        self.noise = nn.Parameter(torch.tensor(noise))

    def forward(self, X, E=1, verbose=False, **kwargs):
        qF1, qU1, pU1 = self.gp1(X, verbose=verbose, **kwargs)
        
        X2 = qF1.mean.T    
        qF2, qU2, pU2 = self.gp2(X2, verbose=verbose, **kwargs)
        
    
        

        F2 = qF2.rsample((E, ))
        noise = torch.nn.functional.softplus(self.noise) #ensure positive
        pY = distributions.Normal(F2, noise)

        return pY, qF1, qU1, pU1, qF2, qU2, pU2
    
    
    def predict(self, X, verbose=False, **kwargs):
        qF1, qU1, pU1 = self.gp1(X, verbose=verbose, **kwargs)
        
        X2 = qF1.mean.T    
        qF2, qU2, pU2 = self.gp2(X2, verbose=verbose, **kwargs)
        
            

        return qF1, qU1, pU1, qF2, qU2, pU2

In [8]:
L1 = 2
M = 300
idx = torch.multinomial(torch.ones(X.shape[0]), num_samples=M, replacement=False)
kernel = NSF_RBF(L=L1, sigma=0.5, lengthscale=1.2)
gp = WSVGP(kernel, M=M, jitter=1e-1)
gp.Lu = nn.Parameter(1e-2*torch.eye(M).expand(L1, M, M).clone())
gp.Z = nn.Parameter(torch.tensor(X[idx]), requires_grad=False)
mu =  torch.squeeze(torch.stack((torch.torch.sin(gp.Z), torch.cos(gp.Z))))
gp.mu = nn.Parameter(mu)


L2 = 3
kernel2 = NSF_RBF(L=L2, sigma=0.5, lengthscale=1.2)
gp2 = WSVGP(kernel2, M=M, jitter=1e-1)
gp2.Lu = nn.Parameter(1e-2*torch.eye(M).expand(L2, M, M).clone())
gp2.Z = nn.Parameter(mu.T, requires_grad=False)

gp2.mu = nn.Parameter(Y[idx].T)

NameError: name 'torch' is not defined